# 02 — TRAM II: Building the MITRE Tactic Label Space

**Goal:** turn TRAM II's sentence-level MITRE ATT&CK *technique* labels into *tactic* labels, and save a clean multi-label dataset.

## Why tactics, not techniques?

MITRE ATT&CK has:

- **~14 tactics** (the *why* of an attack — Initial Access, Persistence, Exfiltration, …)
- **~200 techniques** (the *how* — Phishing, Scheduled Task, DNS Tunneling, …)
- **~600 sub-techniques** (specific variants)

Classifying into tactics is tractable with a few thousand training examples. Classifying into techniques needs an order of magnitude more data per class. We start with tactics and leave techniques as a stretch goal.

## Why multi-label?

A single sentence in a threat report often describes multiple tactics — e.g. "the attacker phished credentials (Initial Access) and used them to move laterally (Lateral Movement)." TRAM II's `multi_label.json` already captures this.

## What this notebook does

1. Clone the TRAM repo
2. Load `multi_label.json`
3. Fetch the ATT&CK Enterprise technique→tactic mapping from MITRE's CTI STIX bundle
4. Convert each sample's techniques into tactic labels (multi-hot)
5. Inspect tactic frequency and co-occurrence
6. Save a `DatasetDict` to `processed/tram_tactics/`

**Runs once.** Skip to notebook 03 on subsequent sessions.

## Step 1 — Clone the TRAM repository

The current canonical repo is `center-for-threat-informed-defense/tram`. The legacy `mitre-attack/tram` is deprecated.

In [1]:
import os
from pathlib import Path

DATA_DIR = Path('data/tram')
REPO_DIR = DATA_DIR / 'repo'
REPO_URL = 'https://github.com/center-for-threat-informed-defense/tram.git'

DATA_DIR.mkdir(parents=True, exist_ok=True)

if REPO_DIR.exists():
    print(f'Already cloned at {REPO_DIR}')
else:
    os.system(f'git clone --depth 1 {REPO_URL} {REPO_DIR}')
    print('Clone complete.')

Cloning into 'data/tram/repo'...


Clone complete.


## Step 2 — Load `multi_label.json`

TRAM II stores labeled data at `data/tram2-data/multi_label.json`. Each record is typically `{"text": "...", "labels": ["T1566", "T1078", ...]}` — sentences with one or more ATT&CK technique IDs.

We normalize the shape defensively since TRAM's JSON schema has drifted across releases.

In [2]:
import json

ML_PATH = REPO_DIR / 'data' / 'tram2-data' / 'multi_label.json'
assert ML_PATH.exists(), f'Expected file not found: {ML_PATH}'

with open(ML_PATH) as f:
    raw = json.load(f)

print(f'Top-level type: {type(raw).__name__}')
if isinstance(raw, dict):
    print(f'Top-level keys: {list(raw.keys())[:10]}')
    # Heuristic: find the list of records
    for k, v in raw.items():
        if isinstance(v, list) and v and isinstance(v[0], dict):
            records = v
            print(f'Using records under key: {k!r}')
            break
    else:
        records = []
else:
    records = raw

print(f'\nTotal records: {len(records)}')
print(f'First record keys: {list(records[0].keys())}')
print(f'First record: {json.dumps(records[0], indent=2)[:500]}')

Top-level type: list

Total records: 19178
First record keys: ['sentence', 'labels', 'doc_title']
First record: {
  "sentence": "title: NotPetya Technical Analysis \u2013 A Triple Threat: File Encryption, MFT Encryption, Credential Theft url: https://www.crowdstrike.com/blog/petrwrap-ransomware-technical-analysis-triple-threat-file-encryption-mft-encryption-credential-theft/ Update: Due to naming convention consistency in the industry, CrowdStrike is now calling this variant of Petya \u2013 NotPetya. ",
  "labels": [],
  "doc_title": "NotPetya Technical Analysis  A Triple Threat File Encryption MFT Encryp


### Extract `(text, techniques)` pairs

Field names have varied across TRAM versions (`text` vs `sentence`, `labels` vs `mappings` vs `attack_ids`). We probe for the right keys and normalize.

In [3]:
import re

TECH_RE = re.compile(r'T\d{4}(?:\.\d{3})?')  # matches T1566 or T1566.001

TEXT_KEYS = ('text', 'sentence', 'content')
LABEL_KEYS = ('labels', 'mappings', 'attack_ids', 'tags', 'techniques')

def get_first(d, keys):
    for k in keys:
        if k in d:
            return d[k]
    return None

def extract_techniques(val):
    """Pull out T-codes from whatever shape the 'labels' field takes."""
    if val is None:
        return []
    if isinstance(val, list):
        out = []
        for item in val:
            if isinstance(item, str):
                out.extend(TECH_RE.findall(item))
            elif isinstance(item, dict):
                # e.g. {"attack_id": "T1566", ...}
                for v in item.values():
                    if isinstance(v, str):
                        out.extend(TECH_RE.findall(v))
        return out
    if isinstance(val, str):
        return TECH_RE.findall(val)
    return []

samples = []
for rec in records:
    text = get_first(rec, TEXT_KEYS)
    techs = extract_techniques(get_first(rec, LABEL_KEYS))
    if text and techs:
        # collapse sub-techniques to parent (T1566.001 -> T1566) for cleaner stats
        parents = sorted({t.split('.')[0] for t in techs})
        samples.append({'text': text, 'techniques': parents})

print(f'Samples with both text and at least one technique: {len(samples)}')
print(f'\nExample:')
print(f'  text:       {samples[0]["text"][:140]}...')
print(f'  techniques: {samples[0]["techniques"]}')

Samples with both text and at least one technique: 4070

Example:
  text:       It spreads to Microsoft Windows machines using several propagation methods, including the EternalBlue exploit for the CVE-2017-0144 vulnerab...
  techniques: ['T1210']


## Step 3 — Fetch the ATT&CK technique→tactic mapping

MITRE publishes ATT&CK as a STIX 2.1 bundle on GitHub. Each `attack-pattern` object lists its `kill_chain_phases`, which are the tactics. We fetch once and cache locally.

The Enterprise bundle is ~30 MB.

In [4]:
import urllib.request

ATTACK_URL = (
    'https://raw.githubusercontent.com/mitre/cti/master/'
    'enterprise-attack/enterprise-attack.json'
)
ATTACK_PATH = DATA_DIR / 'enterprise-attack.json'

if not ATTACK_PATH.exists():
    print('Downloading ATT&CK Enterprise bundle...')
    urllib.request.urlretrieve(ATTACK_URL, ATTACK_PATH)
    print(f'Saved to {ATTACK_PATH} ({ATTACK_PATH.stat().st_size / 1e6:.1f} MB)')
else:
    print(f'Already cached at {ATTACK_PATH}')

Saved to data/tram/enterprise-attack.json (45.1 MB)


In [5]:
with open(ATTACK_PATH) as f:
    attack = json.load(f)

# Build technique_id -> list of tactic shortnames
tech_to_tactics = {}
for obj in attack['objects']:
    if obj.get('type') != 'attack-pattern':
        continue
    if obj.get('revoked') or obj.get('x_mitre_deprecated'):
        continue
    # Find the external MITRE technique ID (T####)
    tech_id = None
    for ref in obj.get('external_references', []):
        if ref.get('source_name') == 'mitre-attack':
            tech_id = ref.get('external_id')
            break
    if not tech_id:
        continue
    tactics = [kcp['phase_name'] for kcp in obj.get('kill_chain_phases', [])
               if kcp.get('kill_chain_name') == 'mitre-attack']
    if tactics:
        tech_to_tactics[tech_id] = tactics

print(f'Techniques mapped: {len(tech_to_tactics)}')
print(f'\nExample: T1566 -> {tech_to_tactics.get("T1566")}')

Techniques mapped: 691

Example: T1566 -> ['initial-access']


### The 14 Enterprise tactics

Collect and sort the full tactic vocabulary from the mapping.

In [6]:
all_tactics = sorted({t for tactics in tech_to_tactics.values() for t in tactics})
tactic2id = {t: i for i, t in enumerate(all_tactics)}

print(f'Tactics ({len(all_tactics)}):')
for t in all_tactics:
    print(f'  {t}')

Tactics (14):
  collection
  command-and-control
  credential-access
  defense-evasion
  discovery
  execution
  exfiltration
  impact
  initial-access
  lateral-movement
  persistence
  privilege-escalation
  reconnaissance
  resource-development


## Step 4 — Project techniques onto tactics

For each TRAM sample, take the union of tactics across all its techniques. Drop samples whose techniques we couldn't map (rare, usually deprecated IDs).

In [7]:
labeled = []
unmapped = 0

for s in samples:
    tactics = set()
    for tech in s['techniques']:
        if tech in tech_to_tactics:
            tactics.update(tech_to_tactics[tech])
    if not tactics:
        unmapped += 1
        continue
    labeled.append({
        'text': s['text'],
        'tactics': sorted(tactics),
    })

print(f'Labeled samples: {len(labeled)}')
print(f'Dropped (no tactic mapping): {unmapped}')
print(f'\nExample:')
print(f'  text:    {labeled[0]["text"][:120]}...')
print(f'  tactics: {labeled[0]["tactics"]}')

Labeled samples: 4070
Dropped (no tactic mapping): 0

Example:
  text:    It spreads to Microsoft Windows machines using several propagation methods, including the EternalBlue exploit for the CV...
  tactics: ['lateral-movement']


## Step 5 — Inspect tactic frequency and co-occurrence

**Frequency** tells us class imbalance (critical for multi-label — rare tactics will have low recall if we don't tune thresholds in notebook 07).

**Co-occurrence** tells us whether tactics behave independently or cluster (e.g. Discovery + Lateral Movement often co-occur). This shapes how we reason about model errors later.

In [8]:
from collections import Counter

tac_freq = Counter()
n_labels_per_sample = Counter()

for ex in labeled:
    tac_freq.update(ex['tactics'])
    n_labels_per_sample[len(ex['tactics'])] += 1

print('Tactic frequencies:')
for tac, count in tac_freq.most_common():
    bar = '#' * int(40 * count / tac_freq.most_common(1)[0][1])
    print(f'  {tac:28s} {count:5d}  {bar}')

print('\nLabels per sample:')
for k in sorted(n_labels_per_sample):
    print(f'  {k} tactic(s): {n_labels_per_sample[k]:5d}')

Tactic frequencies:
  defense-evasion               1916  ########################################
  execution                      850  #################
  privilege-escalation           783  ################
  command-and-control            700  ##############
  persistence                    555  ###########
  discovery                      365  #######
  initial-access                 308  ######
  credential-access              254  #####
  collection                     178  ###
  lateral-movement               176  ###
  exfiltration                    80  #

Labels per sample:
  1 tactic(s):  2770
  2 tactic(s):   799
  3 tactic(s):   286
  4 tactic(s):   155
  5 tactic(s):    48
  6 tactic(s):     8
  7 tactic(s):     3
  10 tactic(s):     1


In [9]:
import numpy as np

# Co-occurrence matrix (tactic i, tactic j) = count of samples where both appear
n_tac = len(all_tactics)
cooc = np.zeros((n_tac, n_tac), dtype=int)
for ex in labeled:
    ids = [tactic2id[t] for t in ex['tactics']]
    for i in ids:
        for j in ids:
            cooc[i, j] += 1

# Normalize by diagonal (self-count) -> P(tactic j | tactic i)
with np.errstate(divide='ignore', invalid='ignore'):
    cond = np.where(cooc.diagonal()[:, None] > 0,
                    cooc / cooc.diagonal()[:, None], 0)

# Show the top 5 conditional partners for each tactic
print('Most likely co-occurring tactics (P(col | row) excluding self):')
for i, tac in enumerate(all_tactics):
    row = cond[i].copy()
    row[i] = 0  # drop self
    top = np.argsort(-row)[:3]
    partners = [f'{all_tactics[j]} ({row[j]:.2f})' for j in top if row[j] > 0]
    print(f'  {tac:28s} -> {", ".join(partners) if partners else "(isolated)"}')

Most likely co-occurring tactics (P(col | row) excluding self):
  collection                   -> credential-access (0.39), discovery (0.12), command-and-control (0.12)
  command-and-control          -> defense-evasion (0.13), execution (0.07), privilege-escalation (0.05)
  credential-access            -> collection (0.28), defense-evasion (0.13), privilege-escalation (0.10)
  defense-evasion              -> privilege-escalation (0.30), persistence (0.19), initial-access (0.09)
  discovery                    -> execution (0.15), defense-evasion (0.08), command-and-control (0.07)
  execution                    -> privilege-escalation (0.20), defense-evasion (0.19), persistence (0.18)
  exfiltration                 -> command-and-control (0.26), collection (0.11), defense-evasion (0.11)
  impact                       -> (isolated)
  initial-access               -> defense-evasion (0.54), privilege-escalation (0.53), persistence (0.53)
  lateral-movement             -> defense-evasion (0.

## Step 6 — Train / validation / test split

Multi-label stratification is harder than single-label (you can't balance every combination). We use a simple random split here; notebook 07 can revisit with `iterative-stratification` if the rare tactics end up missing from validation.

In [10]:
import random

random.seed(42)
shuffled = labeled.copy()
random.shuffle(shuffled)

n = len(shuffled)
n_train = int(0.8 * n)
n_val = int(0.1 * n)

splits = {
    'train': shuffled[:n_train],
    'validation': shuffled[n_train:n_train + n_val],
    'test': shuffled[n_train + n_val:],
}

for name, rows in splits.items():
    seen = Counter(t for ex in rows for t in ex['tactics'])
    missing = set(all_tactics) - set(seen)
    print(f'{name:12s}  {len(rows):5d} samples   missing tactics: {sorted(missing) or "none"}')

train          3256 samples   missing tactics: ['impact', 'reconnaissance', 'resource-development']
validation      407 samples   missing tactics: ['impact', 'reconnaissance', 'resource-development']
test            407 samples   missing tactics: ['impact', 'reconnaissance', 'resource-development']


If any split is missing tactics, increase the sample count (TRAM may have been updated) or switch to iterative stratified sampling. For our purposes, missing tactics in *test* means we can't compute F1 for them — we'll flag this in notebook 07.

## Step 7 — Save as a `DatasetDict`

We store labels as a multi-hot `Sequence(Value('float32'))` of length `len(all_tactics)` — the format HuggingFace's `BertForSequenceClassification` expects for multi-label.

In [11]:
from datasets import Dataset, DatasetDict, Features, Sequence, Value

def to_multihot(tactics):
    vec = [0.0] * len(all_tactics)
    for t in tactics:
        vec[tactic2id[t]] = 1.0
    return vec

features = Features({
    'text': Value('string'),
    'labels': Sequence(Value('float32'), length=len(all_tactics)),
    'tactic_names': Sequence(Value('string')),  # human-readable, for inspection
})

def to_dataset(rows):
    return Dataset.from_dict({
        'text': [r['text'] for r in rows],
        'labels': [to_multihot(r['tactics']) for r in rows],
        'tactic_names': [r['tactics'] for r in rows],
    }, features=features)

ds = DatasetDict({name: to_dataset(rows) for name, rows in splits.items()})

OUT_DIR = Path('processed/tram_tactics')
ds.save_to_disk(str(OUT_DIR))

# Persist the label vocabulary — notebook 07 loads this directly.
with open(OUT_DIR / 'tactic_vocab.json', 'w') as f:
    json.dump({
        'tactics': all_tactics,
        'tactic2id': tactic2id,
    }, f, indent=2)

print(f'Saved to {OUT_DIR}')
print(ds)

Saving the dataset (0/1 shards):   0%|          | 0/3256 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/407 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/407 [00:00<?, ? examples/s]

Saved to processed/tram_tactics
DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'tactic_names'],
        num_rows: 3256
    })
    validation: Dataset({
        features: ['text', 'labels', 'tactic_names'],
        num_rows: 407
    })
    test: Dataset({
        features: ['text', 'labels', 'tactic_names'],
        num_rows: 407
    })
})


## Step 8 — Smoke test: load it back

In [12]:
from datasets import load_from_disk

reloaded = load_from_disk(str(OUT_DIR))
ex = reloaded['train'][0]

print(f'text:         {ex["text"][:140]}...')
print(f'tactic_names: {ex["tactic_names"]}')
print(f'labels shape: {len(ex["labels"])} (matches {len(all_tactics)} tactics)')
print(f'labels sum:   {int(sum(ex["labels"]))}  (= number of positive tactics)')

text:         Some of these nodes operate as part of an encrypted proxy service to prevent attribution by concealing their country of origin and TTPs. ...
tactic_names: ['command-and-control']
labels shape: 14 (matches 14 tactics)
labels sum:   1  (= number of positive tactics)


## Summary

| | What |
|---|---|
| Source | TRAM II `multi_label.json` (MITRE CTID) |
| Label space | ~14 ATT&CK Enterprise **tactics** |
| Label format | multi-hot float vector |
| Splits | 80 / 10 / 10 random |
| Saved to | `processed/tram_tactics/` |

### Things to keep in mind for notebook 07

- **Heavy class imbalance** — a handful of tactics dominate (Discovery, Execution, Defense Evasion). Threshold tuning per class will matter.
- **Strong co-occurrence** — the model should learn label correlations, not just independent classifiers. BCE loss still handles this implicitly.
- **Small dataset** — a few thousand sentences. Expect noisy per-tactic F1; we'll report confidence intervals in notebook 07.

**Next:** notebook 03 downloads an APTnotes sample and extracts text for the active-learning pool.